# 06 — Composite Risk Score and Dashboard Export

**What this notebook establishes:** how the composite risk score blends the three detection layers, the weight-sensitivity robustness check, the resulting review list, and exactly what data each dashboard (Power BI, Streamlit, static page) consumes.

**Estimated runtime:** ~15 seconds

All logic lives in `src/`; this notebook only calls into it and displays results (plan/00_MASTER_PLAN.md sec 5).

## 1. The composite formula

```
composite_risk = 0.50 * percentile_rank(if_score)
               + 0.30 * (rule_flag_count / 5)
               + 0.20 * benford_flag
```
Weights are set a priori by audit judgement (IF as the leading unsupervised signal, rules as corroborating evidence, Benford as population context) — not tuned against the labels. `src/composite.py` implements this once and both Stage 7 (the matrix) and Stage 8 (this export) import it.

In [1]:
import json
import pandas as pd
from pathlib import Path
ROOT = Path('..').resolve()
cs = json.loads((ROOT/'reports/metrics/composite_summary.json').read_text())
print('bands:', cs['band_counts'])
pd.DataFrame(cs['weight_sensitivity'])

bands: {'LOW': 21162, 'MEDIUM': 18681, 'HIGH': 9435, 'CRITICAL': 722}


,variant,w_if,w_rules,w_benford,average_precision,recall_at_1000,precision_at_100
0,Primary,0.50,0.30,0.20,0.181662,0.288030,0.54
1,IF-heavy,0.70,0.20,0.10,0.191371,0.310474,0.55
2,Rules-heavy,0.20,0.60,0.20,0.192499,0.288030,0.54
3,Equal,0.34,0.33,0.33,0.180651,0.288030,0.54
4,No-Benford,0.60,0.40,0.00,0.176742,0.306733,0.52


The Primary weighting is kept as the headline despite two other variants scoring marginally higher AP — disclosed in full rather than switching to whichever variant wins against the labels (`plan/03_STAGES_7-9.md` sec 8.1.2's honesty note).

## 2. The review list

Top-ranked transactions, each with a **suggested audit procedure** derived from its dominant signal — the piece that turns a data-science output into an audit deliverable.

In [2]:
top = pd.read_csv(ROOT/'data/dashboard/top_risk_transactions.csv')
top[['risk_rank','txn_id','amount','composite_risk','risk_band','suggested_procedure']].head(10)

,risk_rank,txn_id,amount,composite_risk,risk_band,suggested_procedure
0,1,TXN-00476376,390.0,0.99921,CRITICAL,Obtain the approval matrix; confirm the author...
1,2,TXN-00791394,390.0,0.99826,CRITICAL,Obtain the approval matrix; confirm the author...
2,3,TXN-00915481,1080.0,0.99712,CRITICAL,Obtain the approval matrix; confirm the author...
3,4,TXN-00256701,450.0,0.99633,CRITICAL,Obtain the approval matrix; confirm the author...
4,5,TXN-00197914,2500.0,0.94000,CRITICAL,Obtain the approval matrix; confirm the author...
5,6,TXN-00796315,540.0,0.93994,CRITICAL,Obtain the approval matrix; confirm the author...
6,7,TXN-00052334,480.0,0.93980,CRITICAL,Obtain the approval matrix; confirm the author...
7,8,TXN-00052332,450.0,0.93976,CRITICAL,Obtain the approval matrix; confirm the author...
8,9,TXN-00284744,5000.0,0.93886,CRITICAL,Obtain the approval matrix; confirm the author...
9,10,TXN-00353443,10000.0,0.93873,CRITICAL,Obtain the approval matrix; confirm the author...


## 3. What each dashboard consumes

| Consumer | Source |
|---|---|
| Power BI (prep only — see `DECISIONS.md` D-0001) | `data/dashboard/*`, `data/processed/dashboard_export.parquet` |
| Streamlit app (`app/streamlit_app.py`) | `data/dashboard/*` only |
| Static instant-load page (`docs/index.html`) | `data/dashboard/*`, rendered at build time by `src/static_dashboard.py` |


In [3]:
manifest = sorted(p.name for p in (ROOT/'data/dashboard').glob('*'))
manifest

['benford_aggregate.csv',
 'benford_by_segment_digit.csv',
 'benford_segments.csv',
 'kpi_summary.json',
 'method_comparison.csv',
 'model_metrics.json',
 'monthly_trend.csv',
 'pr_curve_points.csv',
 'precision_at_k.csv',
 'segment_heatmap.csv',
 'top_risk_5000.csv',
 'top_risk_transactions.csv']